In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import os
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import Counter

# =====================================================================
# 1. PATHS & SETUP
# =====================================================================
# HAM10000 dataset paths matching your Kaggle input panel
HAM_IMAGE_DIR_PART1 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
HAM_IMAGE_DIR_PART2 = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
HAM_META_PATH       = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'

# Checkpoint directory matching your uploaded dataset path
CKPT_DIR = '/kaggle/input/datasets/ittisamurtunib/resnet-18'

# Output
OUTPUT_DIR = '/kaggle/working/HAM_Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Dataset __getitem__ method implementation for dual-part image retrieval
def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img_path_part1 = os.path.join(self.image_dir_part1, row['image_id'] + '.jpg')
    img_path_part2 = os.path.join(self.image_dir_part2, row['image_id'] + '.jpg')
    
    if os.path.exists(img_path_part1):
        img_path = img_path_part1
    else:
        img_path = img_path_part2
        
    img = Image.open(img_path).convert('RGB')
    # ... rest of your dataset code

# =====================================================================
# 2. LOAD & FILTER HAM10000 TO 3 CLASSES (BKL, MEL, NV)
# =====================================================================
meta = pd.read_csv(HAM_META_PATH)

# Mapping to match your three classes
def map_dx(dx):
    if dx == 'mel':
        return 'MEL'
    elif dx == 'nv':
        return 'NV'
    elif dx == 'bkl':
        return 'BKL'
    return None

meta['label'] = meta['dx'].apply(map_dx)
meta_filtered = meta[meta['label'].notnull()].copy()
meta_filtered['image'] = meta_filtered['image_id']  # rename to match dataloader

print(f"Total HAM10000 images kept: {len(meta_filtered)}")
print("Class distribution in HAM10000:")
print(meta_filtered['label'].value_counts())

# Save the test CSV
ham_test_csv = os.path.join(OUTPUT_DIR, 'ham_test.csv')
meta_filtered[['image', 'label']].to_csv(ham_test_csv, index=False)

# =====================================================================
# 3. DATALOADER (SAME TRANSFORM AS YOUR TRAINING SCRIPT)
# =====================================================================
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def __getitem__(self, idx):
    row = self.df.iloc[idx]
    image_id = str(row['image_id'])

    # Search candidates across both folders (with .jpg and .JPG extensions)
    possible_paths = [
        os.path.join(self.image_dir_part1, f"{image_id}.jpg"),
        os.path.join(self.image_dir_part2, f"{image_id}.jpg"),
        os.path.join(self.image_dir_part1, f"{image_id}.JPG"),
        os.path.join(self.image_dir_part2, f"{image_id}.JPG"),
    ]

    img_path = next((p for p in possible_paths if os.path.exists(p)), None)

    if img_path is None:
        raise FileNotFoundError(f"Image {image_id} not found in part_1 or part_2 directories.")

    img = Image.open(img_path).convert('RGB')
    if self.transform:
        img = self.transform(img)

    raw_label = row['dx'] if 'dx' in row else row['label']
    label = self.class_to_idx[raw_label]

    return img, label

test_df_ham = pd.read_csv(ham_test_csv)
# Pass meta_filtered directly instead of reading ham_test_csv
test_ds = SimpleDataset(
    df=meta_filtered, 
    image_dir_part1=HAM_IMAGE_DIR_PART1, 
    image_dir_part2=HAM_IMAGE_DIR_PART2, 
    transform=test_transform
)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

# =====================================================================
# 4. MODEL DEFINITIONS (MUST MATCH YOUR SAVED ARCHITECTURE)
# =====================================================================
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class ResNetClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet = models.resnet18(weights=None)  # weights will come from checkpoint
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        features = self.encoder(x)
        features = features.squeeze(-1).squeeze(-1)
        return self.classifier(features)

# =====================================================================
# 5. EVALUATION FUNCTION
# =====================================================================
def evaluate_model(model, loader, classes):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
    mel_recall = report['MEL']['recall']
    mel_precision = report['MEL']['precision']
    mel_f1 = report['MEL']['f1-score']
    cm = confusion_matrix(all_labels, all_preds)
    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'mel_recall': mel_recall,
        'mel_precision': mel_precision,
        'mel_f1': mel_f1,
        'report': report,
        'confusion_matrix': cm.tolist(),
        'predictions': all_preds,
        'true_labels': all_labels
    }

# =====================================================================
# 6. DYNAMIC CHECKPOINT FINDER & EVALUATION
# =====================================================================
import glob

all_pth_files = glob.glob('/kaggle/input/**/*.pth', recursive=True)

ssl_ckpt = next((f for f in all_pth_files if 'ssl' in f.lower()), None)
resnet_ckpt = next((f for f in all_pth_files if 'resnet' in f.lower()), None)

print(f"Found SSL Checkpoint: {ssl_ckpt}")
print(f"Found ResNet Checkpoint: {resnet_ckpt}")

results = {}
class_names = ['BKL', 'MEL', 'NV']

# ---- 6a. SSL + Rebalance ----
if ssl_ckpt and os.path.exists(ssl_ckpt):
    print(f"\nLoading SSL+Rebalance checkpoint: {ssl_ckpt}")
    encoder = SimpleEncoder()
    model = SSLClassifier(encoder, len(class_names)).to(device)
    model.load_state_dict(torch.load(ssl_ckpt, map_location=device))
    res = evaluate_model(model, test_loader, class_names)
    results['SSL_Rebalance'] = res
    
    print("\n" + "=" * 50)
    print("SSL + Rebalance on HAM10000")
    print("=" * 50)
    print(f"Accuracy: {res['accuracy']:.3f}")
    print(f"Macro-F1: {res['macro_f1']:.3f}")
    print(f"MEL Recall: {res['mel_recall']:.1%}")
    print(f"MEL Prec: {res['mel_precision']:.1%}")
    print(f"MEL F1: {res['mel_f1']:.3f}")
    print("\nClassification Report:")
    print(classification_report(res['true_labels'], res['predictions'], target_names=class_names, digits=3))
else:
    print("\nNo SSL checkpoint file found in /kaggle/input")

# ---- 6b. ResNet-18 baseline ----
if resnet_ckpt and os.path.exists(resnet_ckpt):
    print(f"\nLoading ResNet-18 checkpoint: {resnet_ckpt}")
    model_res = ResNetClassifier(len(class_names)).to(device)
    model_res.load_state_dict(torch.load(resnet_ckpt, map_location=device))
    res = evaluate_model(model_res, test_loader, class_names)
    results['ResNet18'] = res
    
    print("\n" + "=" * 50)
    print("ResNet-18 on HAM10000")
    print("=" * 50)
    print(f"Accuracy: {res['accuracy']:.3f}")
    print(f"Macro-F1: {res['macro_f1']:.3f}")
    print(f"MEL Recall: {res['mel_recall']:.1%}")
    print(f"MEL Prec: {res['mel_precision']:.1%}")
    print(f"MEL F1: {res['mel_f1']:.3f}")
    print("\nClassification Report:")
    print(classification_report(res['true_labels'], res['predictions'], target_names=class_names, digits=3))
else:
    print("\nNo ResNet checkpoint file found in /kaggle/input")

# =====================================================================
# 7. SAVE RESULTS
# =====================================================================
# Convert numpy types to Python types for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    else:
        return obj

results_serializable = convert_to_serializable(results)

with open(os.path.join(OUTPUT_DIR, 'ham10000_external_results.json'), 'w') as f:
    json.dump(results_serializable, f, indent=2)

print(f"\n✅ Results saved to {OUTPUT_DIR}/ham10000_external_results.json")
print("\n📊 Summary of external validation:")
for model_name, res in results.items():
    print(f"  {model_name}: Acc={res['accuracy']:.3f}, MF1={res['macro_f1']:.3f}, MEL Recall={res['mel_recall']:.1%}")

Device: cuda
Total HAM10000 images kept: 8917
Class distribution in HAM10000:
label
NV     6705
MEL    1113
BKL    1099
Name: count, dtype: int64
Found SSL Checkpoint: /kaggle/input/datasets/ittisamurtunib/ssl-3seed-pth/ssl_rebalance_seed2024_finetuned.pth
Found ResNet Checkpoint: /kaggle/input/datasets/ittisamurtunib/resnet-18-pth/best_resnet18_seed123.pth

Loading SSL+Rebalance checkpoint: /kaggle/input/datasets/ittisamurtunib/ssl-3seed-pth/ssl_rebalance_seed2024_finetuned.pth

SSL + Rebalance on HAM10000
Accuracy: 0.718
Macro-F1: 0.459
MEL Recall: 6.6%
MEL Prec: 33.9%
MEL F1: 0.111

Classification Report:
              precision    recall  f1-score   support

         BKL      0.312     0.601     0.411      1099
         MEL      0.339     0.066     0.111      1113
          NV      0.862     0.846     0.854      6705

    accuracy                          0.718      8917
   macro avg      0.504     0.505     0.459      8917
weighted avg      0.729     0.718     0.706      8917


Lo